In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix,accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split,GridSearchCV,RandomizedSearchCV
from sklearn.feature_selection import SelectKBest, f_classif,chi2
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from lightgbm import LGBMClassifier




In [3]:
import json
import numpy as np
import pandas as pd

In [4]:
with open('train_part1.json', 'r') as f:
    data = json.load(f)
# Prepare features and labels

X = []
y = []

for record in data:
# Concatenate image and text embeddings
    features = record['image_embedding'] + record['text_embedding']
    X.append(features)
    y.append(record['label'])

X = np.array(X)  # Shape: (1530, 1024)
y = np.array(y)  # Shape: (1530,)

In [5]:
feature_names = [f"feature_{i}" for i in range(X.shape[1])]

# Build dataframe
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

In [10]:
X = df.drop(['target'], axis=1)

In [11]:
y = df['target']

In [12]:
X_train, X_val, y_train, y_val = train_test_split(
X, y, test_size=0.3, random_state=42, stratify=y
)

In [ ]:
# Split a small validation set from training data
lgbm_clf = LGBMClassifier(
    boosting_type='gbdt',
    objective='binary',          # or 'multiclass' for multi-class problems
    metric='f1_macro',                # or 'multi_logloss' for multiclass
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,                # -1 means no limit
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# Fit with eval_set
lgbm_clf.fit(
    X_train,y_train,
)


[LightGBM] [Info] Number of positive: 143, number of negative: 928
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.049881 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 261120
[LightGBM] [Info] Number of data points in the train set: 1071, number of used features: 1024
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.133520 -> initscore=-1.870187
[LightGBM] [Info] Start training from score -1.870187
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [21]:
from sklearn.metrics import f1_score


In [22]:
y_pred = lgbm_clf.predict(X_val)
f1_macro = f1_score(y_val, y_pred, average='macro')
print(f"Validation F1 Macro Score: {f1_macro:.4f}")

Validation F1 Macro Score: 0.6205


In [23]:
with open('test.json', 'r') as f:
    test_data = json.load(f)

X_test = []
test_ids = []

for record in test_data:
    features = record['image_embedding'] + record['text_embedding']
    X_test.append(features)
    test_ids.append(record['id'])

X_test = np.array(X_test)
predictions = lgbm_clf.predict(X_test)

c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [24]:
# 7. Create submission file

submission = pd.DataFrame({
'row_id': test_ids,
'target': predictions
})
submission.to_csv('lgbm_baseline.csv', index=False)
print("Submission file created!")

Submission file created!


In [ ]:
param_grid = {
    'num_leaves': [50,63, 80],
    'max_depth': [4, 5, 6,7],
    'learning_rate': [0.01, 0.02,.03,.04],
    'n_estimators': [1000,1100,1200],
    'subsample': [0.75, 0.8, 0.85,.9],
    'colsample_bytree': [.5,.6,.55],
    'reg_alpha': [0.4, 0.5, 0.55],     # L1 regularization
    'reg_lambda': [.4, 0.5, 0.55],    # L2 regularization
    'min_child_samples': [25, 30, 35]
}


In [26]:
random_search = RandomizedSearchCV(
    estimator=lgbm_clf,          # or lgbm_reg
    param_distributions=param_grid,
    n_iter=30,                   # number of parameter combinations to try
    scoring='f1_macro',           # or 'neg_mean_squared_error' for regression
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best Parameters:", random_search.best_params_)
print("Best Score:", random_search.best_score_)


Fitting 5 folds for each of 30 candidates, totalling 150 fits
[LightGBM] [Info] Number of positive: 143, number of negative: 928
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.033805 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 261120
[LightGBM] [Info] Number of data points in the train set: 1071, number of used features: 1024
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.133520 -> initscore=-1.870187
[LightGBM] [Info] Start training from score -1.870187
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

In [30]:
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_val)
f1_macro = f1_score(y_val, y_pred, average='macro')
print(f"Validation F1 Macro Score (Best Estimator): {f1_macro:.4f}")


Validation F1 Macro Score (Best Estimator): 0.6007


In [28]:
with open('test.json', 'r') as f:
    test_data = json.load(f)

X_test = []
test_ids = []

for record in test_data:
    features = record['image_embedding'] + record['text_embedding']
    X_test.append(features)
    test_ids.append(record['id'])

X_test = np.array(X_test)
predictions = best_model.predict(X_test)

c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [29]:
# 7. Create submission file

submission = pd.DataFrame({
'row_id': test_ids,
'target': predictions
})
submission.to_csv('lightgbm_param_grid.csv', index=False)
print("Submission file created!")

Submission file created!
